# Demo: Baseline Extraction

A prompt-only Research Agent attempts to emit Turtle directly, without `DatasetMiddleware`.

---

- **What condition does this notebook represent?** A baseline in which the agent receives task guidance but no dataset-backed tool surface, no middleware-owned RDF working memory, and no middleware-controlled serialization path.
- **What is the point of this baseline?** To establish what direct prompting can and cannot do before we add capability-gated middleware.
   - Can the agent produce parseable Turtle at all?
   - If it can, how much semantic overlap does it have with a target graph?
   - If it cannot, syntax failure itself is part of the result.
- **What is intentionally still present?** Deepagents' default behavior and `MinistralPromptSuffixMiddleware` remain so that we do not accidentally degrade the model for reasons unrelated to RDF extraction.
- **Where does this notebook fit in the series?** It is the negative control for [demo-dataset-middleware.ipynb](./demo-dataset-middleware.ipynb), where the Research Agent receives dataset tools and a middleware-owned RDF side channel.

## Where does this notebook fit in the series?

| Condition | Middleware | Notebook |
|-----------|-----------|----------|
| **Baseline (prompt-only)** | None | **this notebook** |
| Dataset middleware | `DatasetMiddleware` | [demo-dataset-middleware.ipynb](./demo-dataset-middleware.ipynb) |
| Vocabulary middleware | `DatasetMiddleware` + `RDFVocabularyMiddleware` | [demo-vocabulary-middleware.ipynb](./demo-vocabulary-middleware.ipynb) |
| Domain-Specific TBox |  `DatasetMiddleware` + `RDFVocabularyMiddleware` | [./demo-vocabulary-middleware-ext.ipynb](./demo-vocabulary-middleware-ext.ipynb) |

## 1. Setup

We first filter a LangChain-internal warning that is unrelated to the experiment.

In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
    category=UserWarning,
)

### 1.1. Bring your own Model

This notebook uses [LangChain's OpenAI integration](https://docs.langchain.com/oss/python/integrations/chat/openai) for [a self-hosted model provider](https://docs.langchain.com/langsmith/playground-model-providers#openai-compatible-endpoint), but you can use any of [LangChain's supported providers](https://docs.langchain.com/oss/python/integrations/providers/all_providers)
What's important is that Deepagents requires a chat model capable of tool invocations.

_NOTE:_ [My Ministral model](https://huggingface.co/mistralai/Ministral-3-14B-Reasoning-2512) recommends adding specific suffix to its prompt. In this repository we also provide `MinistralPromptSuffixMiddleware` to append it so that it doesn't catch people by surprise if they are using other models. Instead, just modify the prompt directly to match your specific model's requirements. Additionally, the model's documentation indicates that one should use `temperature=1` over `temperature=0`, so we follow that guidance as well.

In [2]:
import os

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    seed=42,
    temperature=1,
)

## 2. The Demonstration

This notebook uses the same task family as the middleware demos, but keeps the runtime condition deliberately narrow.

### 2.1. Prompt Setup

The shared prompt is assembled from reusable components in [`demo_utils.py`](./demo_utils.py).
Those components include RDF modeling guidance, vocabulary-selection guidance, and a shared stop condition.

Methodologically, these additions should be understood as **tool-agnostic guidance extracted from middleware prompts**.
The goal is to reduce prompt asymmetry between conditions, not to silently tune the baseline or recreate middleware capabilities.

What this baseline **does** receive:

- Domain-relevant instructions about RDF modeling.
- Guidance to prefer grounded, stable, documented modeling choices.
- A direct instruction to return exactly one Turtle block.

What this baseline **does not** receive:

- `DatasetMiddleware` tools such as `add_triples` or `serialize_dataset`.
- A middleware-owned RDF dataset that can accumulate state across reasoning steps.
- A structured serialization path that can prevent malformed final Turtle.

That distinction matters for interpretation: if `DatasetMiddleware` produces more parseable outputs, that is part of the middleware effect rather than evidence that the baseline was unfairly under-prompted.

### 2.2. Prompt Limitations

Prompt guidance can encourage better modeling choices, but it cannot replace missing affordances.
This baseline still asks the Research Agent to compose Turtle directly in its final answer.

As a result, the evaluation has a hard first gate:

1. Is there exactly one Turtle block?
2. Does that block parse as RDF?

If the answer to either question is no, semantic graph metrics are unavailable for that run.

In [3]:
from typing import Final

from demo_utils import CORE_PROMPT, DATASET_TIPS, STOPPING_CRITERIA, VOCABULARY_TIPS

# Reduce prompt asymmetry without recreating middleware-only capabilities.
SYSTEM_PROMPT: Final[str] = (
    CORE_PROMPT + DATASET_TIPS + VOCABULARY_TIPS + STOPPING_CRITERIA
)

### 2.3. Agent Task Definition

We keep the extraction task itself fixed across the series so that capability changes come from middleware composition rather than from changing the problem statement.

The task asks the Research Agent to represent a short natural-language passage as RDF.
In this baseline condition, the agent must do that by emitting Turtle directly in its final answer.

In [4]:
from demo_utils import TASK

print(TASK)


Please assist me in representing the subject matter of the following text as an RDF graph.
Please reuse established RDF, RDFS, OWL, or other appropriate standard terms
when they fit the source. Please use <urn:ex:> as the base only for IRIs that
you genuinely need to mint as part of your response. Do not invent helper
relations or organizing abstractions unless they are needed to represent an
explicit claim from the text. Be sure to label any new terms or properties that
you mint so that they are human readable.

```text
John is a person. Modern people are classified as _homo sapiens_. Apparently,
homo sapiens falls under the subtribe of Hominina. Every Hominidae is a
Haplorhini, and now the names don't even sound like they're related. Then
we get to primates, which can be controversial for some people, for some reason.
Nobody argues with the idea that primates are mammals, yet some people take
umbrage with the idea that _homo sapiens_ is an animal. Oh, Hominidae contains
Hominina, to

#### 2.3.1 Evaluation Contract

For this notebook, evaluation proceeds in two layers.

First, the final answer must be machine-usable:

1. The response should contain exactly one `text/turtle` fenced block.
2. That Turtle should parse successfully.

Second, if parsing succeeds, we compare the resulting graph against a target graph.
Best-practice documentation triples such as `rdfs:label` and `rdfs:comment` are acceptable extras rather than automatic errors.

In [5]:
from demo_utils import expected_graph
from rdflib import Graph

print(expected_graph.serialize(format="turtle"))

@prefix ex: <urn:ex:> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Haplorhini a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Primate .

ex:Hominidae a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Haplorhini .

ex:Hominina a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Hominidae .

ex:HomoSapiens a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Hominina .

ex:Mammal a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Animal .

ex:Primate a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Mammal .

ex:John a ex:Person .

ex:Person rdfs:subClassOf ex:HomoSapiens ;
    owl:equivalentClass ex:HomoSapiens .




### 2.4. Example Run

This notebook keeps the run presentation deliberately simple.
Unlike the middleware conditions, the baseline has no middleware-owned RDF state to inspect after the fact.
The main artifact is therefore the Research Agent's final answer, together with the question of whether that answer contains one parseable Turtle block.

That difference in observability matters for the series.
The later notebooks can compare traces, stored datasets, and graph-level metrics, while this baseline must first clear a stricter operational hurdle: direct final-answer serialization.

In [6]:
from deepagents import create_deep_agent
from langgraph.errors import GraphRecursionError
from rdflib_reasoning.middleware.continuation_guard_middleware import (
    ContinuationGuardMiddleware,
)
from rdflib_reasoning.middleware.ministral_middleware import (
    MinistralPromptSuffixMiddleware,
)
from rdflib_reasoning.middleware.tracing_notebook import LiveNotebookTrace

with LiveNotebookTrace(heading="Agent Activity") as trace:
    agent = trace.attach(
        create_deep_agent(
            model=llm,
            system_prompt=SYSTEM_PROMPT,
            middleware=[
                MinistralPromptSuffixMiddleware(),
                ContinuationGuardMiddleware(),
            ],
        )
    )

    RUN_INPUT = {"messages": [{"role": "user", "content": TASK}]}
    RUN_CONFIG = {"recursion_limit": 20}
    try:
        agent_response = agent.invoke(RUN_INPUT, config=RUN_CONFIG)
    except GraphRecursionError:
        display(trace.sink.snapshot())
        raise

# Agent Activity

## Turn 1

### Model Input Summary

- `system`: You are a research assistant specialized in knowledge graphs and the Semantic Web. Task objective: - Represent the provided source text as RDF, grounded in explicit claims from that text. - Do not introduce uncertain or speculative facts...
- `human`: Please assist me in representing the subject matter of the following text as an RDF graph. Please reuse established RDF, RDFS, OWL, or other appropriate standard terms when they fit the source. Please use <urn:ex:> as the base only for I...

### Final Response

> ინგი
> - قاموس RDF-ზე რ{W
>
>  weaving the source text into a faithful RDF representation. I will analyze the text for clear fact-based assertions and represent those with RDF, RDFS, or OWL terms where appropriate.
> - I will only mint IRIs under <urn:ex:> if absolutely necessary and will document them with `rdfs:label` and `rdfs:comment`.
> - I will ensure that the representation remains grounded in explicit claims from the text and will not include uncertain or speculative facts.
>
> **Step 1: Identify Entities and Relationships**
> The text contains the following key entities and relationships:
> 1. John is a person.
> 2. Modern people (humans) are classified as *homo sapiens*.
> 3. *homo sapiens* belongs to the subtribe Hominina.
> 4. Hominidae is a category that includes Hominina.
> 5. Every Hominidae is a member of Haplorhini.
> 6. Hominidae contains Hominina (redundant with point 4, but included in the text).
> 7. Primates are mammals, and some might debate this categorization.
> 8. *homo sapiens* is also an animal, which some people reject.
>
> **Step 2: Choose Appropriate RDF, RDFS, or OWL Predicates**
> I will use:
> - `rdf:type` for instance or class relationships.
> - `rdfs:label` for human-readable labels.
> - `rdfs:comment` for additional context.
> - Prefer existing standard terms like `skos:broader`, `skos:narrower`, or `owl:subClassOf` where applicable.
> - For debated or emotionally charged claims (e.g., "primates are mammals" or "*homo sapiens* is an animal"), I will represent the claim as stated but avoid modeling emotional or theological views unless explicitly required.
>
> **Step 3: Represent Explicit Claims**
> I will represent:
> 1. John as an instance of a *Person*.
> 2. Modern people as instances of *Homo sapiens*.
> 3. The classification hierarchy:
>    - *Homo sapiens* is a subtribe of Hominina.
>    - Hominidae is a broader category for Hominina.
>    - Hominidae is a subclass of Haplorhini.
>    - Hominidae contains Hominina (redundant, but included).
> 4. Primates as mammals, with a note about controversy.
> 5. *Homo sapiens* as an animal, with a note about disagreement.
>
> **Step 4: Mint Minimal, Documented URIs**
> If I need to mint IRIs, I will define them under <urn:ex:> and provide labels and comments.
>
> **Step 5: Finalize RDF Representation**
> I will ensure that only correct, fact-based facts are represented. Avoid speculative or debatable points unless explicitly stated.
>
> Let me proceed to represent the content in Turtle format, grounding it strictly in the source text and using standard RDF, RDFS, OWL terms wherever they fit. If necessary, I will create minimal, documented URIs under <urn:ex:>.
>
> ### let's get started
> I will start by listing out the statements to validate and model in RDF.
>
> ...
>     The term *Homo sapien*s is used as a scientific name, it's best represented as a label or a class.
>     "Modern people are classified as *homo sapiens*" implies that modern people are instances of the class Homo sapiens.
>     "homo sapiens falls under the subtribe of Hominina" implies that Hominina is a broader category for Homo sapiens.
>     
>     For taxonomic relationships, the following terms might be useful:
>     - skos:broader / skos:narrower from SKOS for hierarchical relationships.
>     - owl:subClassOf for subclass relationships in ontologies.
>     - ro:Class for classes in a taxonomic hierarchy (if appropriate).
>     
>     However, if I'm to represent this hierarchy without external vocabularies, I can also use rdfs:subClassOf.
>     
>     Since the text is about biology and taxonomy, using a taxonomy-appropriate vocabulary like skos or ro (from the Real World ontology or similar) would be best, but it's not clear if the user is okay with external vocabularies.
>     
>     I will try to use rdfs:subClassOf for simplicity, but note that skos or other vocabularies are more precise for taxonomic hierarchies.
>     
>     The statement "Emotions and theological views aside, can we formalize this?" suggests a commentary, but it doesn't introduce a new fact, so I will not model it.
>     
>     The text says "Then we get to primates, which can be controversial for some people, for some reason. Nobody argues with the idea that primates are mammals, yet some people take umbrage with the idea that *homo sapiens* is an animal."
>     
>     This seems to imply:
>     - Primates are mammals (non-controversial, per the text).
>     - *Homo sapiens* is an animal (controversial for some).
>     
>     I will represent:
>     - That *Homo sapiens* is a subclass of Animal.
>     - That Primates are a subclass of Mammalia.
>     - I will not model the controversy or reasons for it, unless explicitly requested.
>
> Let's proceed to write the RDF in Turtle.
> I will define:
> - John as a Person.
> - Modern people as instances of Homo sapiens.
> - The taxonomic hierarchy using rdfs:subClassOf and skos:broader/skos:narrower if I decide to use SKOS. For now, I'll use rdfs:subClassOf and clarify in the comment.
>
> Let me draft the Turtle:
> ```turtle
> <urn:ex:John> a <urn:ex:Person> ;
>     rdfs:label "John" .
>
> <urn:ex:Homo_sapiens> a owl:Class ;
>     rdfs:label "Homo sapiens" ;
>     rdfs:comment "Modern humans" .
>
> <urn:ex:Person> a owl:Class ;
>     rdfs:label "Person" ;
>     rdfs:subClassOf <urn:ex:Homo_sapiens> .
>
> <urn:ex:Homo_sapiens> a owl:Class ;
>     rdfs:label "Homo sapiens" ;
>     rdfs:subClassOf <urn:ex:Hominina> ;
>     rdfs:comment "Class for modern humans" .
>
> <urn:ex:Hominina> a owl:Class ;
>     rdfs:label "Hominina" ;
>     rdfs:comment "Subtribe containing Homo sapiens" .
>
> <urn:ex:Hominidae> a owl:Class ;
>     rdfs:label "Hominidae" ;
>     rdfs:comment "Family containing Hominina" ;
>     rdfs:subClassOf <urn:ex:Haplorhini> ;
>     rdfs:subClassOf <urn:ex:HominidaeિયაHominina bå.> a owl:Class ;
>     rdfs:label "Hominina" ;
>     rdfs:comment "Subtribe under Hominidae" ;
>
> <urn:ex:Hominidae> a owl:Class ;
>     rdfs:label "Hominidae" ;
>     rdfs:subClassOf <urn:ex:Haplorhini> ;
>     rdfs:comment "Family containing Hominina" .
>
> <urn:ex:Haplorhini> a owl:Class ;
>     rdfs:label "Haplorhini" ;
>     rdfs:comment "Taxonomic group containing Hominidae" .
>
> <urn:ex:Primates> a owl:Class ;
>     rdfs:label "Primates" ;
>     rdfs:subClassOf <urn:ex:Mammalia> ;
>     rdfs:comment "Order containing hominids among others" .
>
> <urn:ex:Mammalia> a owl:Class ;
>     rdfs:label "Mammals" ;
>     rdfs:comment "Class of animals that nurse their young" .
>
> <urn:ex:Animal> a owl:Class ;
>     rdfs:label "Animal" ;
>     rdfs:comment "Common ancestor group for mammals, etc." .
>
> <urn:ex:Homo_sapiens> a owl:Class ;
>     rdfs:label "Homo sapiens" ;
>     rdfs:subClassOf <urn:ex:Animal> ;
>     rdfs:comment "Modern humans are also animals" .
>
> <urn:ex:Person> rdfs:subClassOf <urn:ex:Homo_sapiens> .
>
> <urn:ex:John> a <urn:ex:Person> .
>
> <urn:ex:Modern_people> a owl:Class ;
>     rdfs:label "Modern people" ;
>     rdfs:subClassOf <urn:ex:Homo_sapiens> .
>
> # Additional note about controversy in the text, but I won't model it as a fact
> ```
> Let's clean this up and ensure that the hierarchy is correct.
>
> - Hominidae contains Hominina: This is a subclass relationship, so rdfs:subClassOf is appropriate.
> - Hominidae is a subclass of Haplorhini: Also appropriate.
> - Primoates are mammals: So, Primates owl:equivalentClass or rdfs:subClassOf Mammalia. I'll use rdfs:subClassOf for simplicity.
> - Homo sapiens is an animal: So, rdfs:subClassOf Animal.
>
> Let me re-draft with correct hierarchy and minimal URI minting.
>
> I will break it into clear sections and ensure documentation is added for any minted URIs.
>
> Let me create the Turtle, carefully checking the hierarchy and documentation.
>
> ###let's represent the content in Turtle with the above approach, noting only that the controversy will not be modeled as a fact. The text says that some people have difficulty with these classifications, but the RDF representation will only encode the "canon" view.
>
> - I will represent John as an instance of Person.
> - I will represent Homo sapiens as a class, instance of owl:Class.
> - I will create the taxonomic hierarchy using rdfs:subClassOf.
>
> Let me draft it now, noting that I will not introduce any new properties, only utilize existing and documented ones.
>
> ```turtle
> @prefix ex: <urn:ex:> .
> @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
> @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
> @prefix owl: <http://www.w3.org/2002/07/owl#> .
> @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
>
>
>
> ex:John a ex:Person ;
>     rdfs:label "John" .
>
> ex:Person a owl:Class ;
>     rdfs:label "Person" ;
>     rdfs:comment "A Modern human." ;
>     rdfs:subClassOf ex:Homo_sapiens .
>
> ex:Homo_sapiens a owl:Class ;
>     rdfs:label "Homo sapiens" ;
>     rdfs:comment "The scientific classification for modern humans." ;
>     rdfs:subClassOf ex:Hominina .
>
> ex:Hominina a owl:Class ;
>     rdfs:label "Hominina" ;
>     rdfs:comment "A subtribe containing Homo sapiens." .
>
> ex:Hominidae a owl:Class ;
>     rdfs:label "Hominidae" ;
>     rdfs:comment "The family containing Hominina." ;
>     owl:equivalentClass ex:Hominidae .
>     # Use owl:equivalentClass to represent that Hominidae is the same as HominidaeIterator#EP6
>
> ex:Hominidae a rdfs:subClassOf ex:Haplorhini .
>
> ex:Haplorhini a owl:Class ;
>     rdfs:label "Haplorhini" ;
>     rdfs:comment "A taxonomic group containing Hominidae." .
>
> ex:Primates a owl:Class ;
>     rdfs:label "Primates" ;
>     rdfs:comment "An order of mammals containing Hominidae." ;
>     rdfs:subClassOf ex:Mammalia .
>
> ex:Mammalia a owl:Class ;
>     rdfs:label "Mammalia" ;
>     rdfs:comment "Class of mammals." ;
>
> ex:Animal a owl:Class ;
>     rdfs:label "Animal" ;
>     rdfs:comment "A living organism capable of sensation and voluntary movement."  ;
>     rdfs:subClassOf ex:Animal .
>
> ex:Homo_sapiens a rdfs:subClassOf ex:Animal .
>
> # Modern people are classified as Homo sapiens
> ex:Modern_people a owl:Class ;
>     rdfs:label "Modern people" ;
>     rdfs:comment "Refers to people living today, classified as Homo sapiens." ;
>     rdfs:subClassOf ex:Homo_sapiens .
>
> ex:John a ex:Person .
>
> # Additional note about the text content, but not modeled as fact
> ```
> Let me check the hierarchy and comments to ensure that it is correct and that each statement in the text is represented.
>
> The text says:
> > Modern people are classified as *homo sapiens*.
> - Represented by: `ex:Modern_people rdfs:subClassOf ex:Homo_sapiens`.
>
> > *homo sapiens* falls under the subtribe of Hominina.
> - Represented by: `ex:Homo_sapiens rdfs:subClassOf ex:Hominina`.
>
> > Every Hominidae is a Haplorhini
> - Represented by: `ex:Hominidae rdfs:subClassOf ex:Haplorhini` (using owl:equivalentClass is not correct, as per my earlier draft; I will instead use rdfs:subClassOf).
>
> > Oh, Hominidae contains Hominina, too.
> - Represented by: `ex:Hominina rdfs:subClassOf ex:Hominidae` (subClassOf is the correct direction for "contains").
>
> > Primates are mammals.
> - Represented by: `ex:Primates rdfs:subClassOf ex:Mammalia`.
>
> > *homo sapiens* is an animal.
> - Represented by: `ex:Homo_sapiens rdfs:subClassOf ex:Animal`.
>
> I need to correct the direction of the last two subclass relations.
>
> Also, the text says "Hominidae contains Hominina", so Hominina is a subclass of Hominidae, not the other way around.
>
> Similarly, "Hominidae is a Haplorhini" means Hominidae is a subclass of Haplorhini.
>
> Let me fix the hierarchy in the Turtle.
>
> Let me first get the order correct for "Hominidae contains Hominina" (i.e., Hominina is a subclass of Hominidae).
>
> Also, the text says: "Every Hominidae is a Haplorhini", which means owl:Class Hominidae is a subclass of owl:Class Haplorhini.
>
> So:
>
> - Hominidae subclasses Haplorhini: `ex:Hominidae a rdfs:subClassOf ex:Haplorhini`
> - Hominina subclasses Hominidae: `ex:Hominina a rdfs:subClassOf ex:Hominidae`
>
> Let me updates those lines.
>
> Also, for Primates, the text says "primates are mammals", so Primates is a subclass of Mammalia.
>
> And for Animal, "Homo sapiens is an animal", so Homo sapiens subclass Animal.
>
> Let me rewrite the Turtle correctly and then double-check myself.
>
> ```turtle
> @prefix ex: <urn:ex:> .
> @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
> @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
> @prefix owl: <http://www.w3.org/2002/07/owl#> .
> @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
>
> ex:John a ex:Person ;
>     rdfs:label "John" .
>
> ex:Person a owl:Class ;
>     rdfs:label "Person" ;
>     rdfs:comment "A Modern human." ;
>     rdfs:subClassOf ex:Homo_sapiens .
>
> ex:Homo_sapiens a owl:Class ;
>     rdfs:label "Homo sapiens" ;
>     rdfs:comment "The scientific classification for modern humans." ;
>     rdfs:subClassOf ex:Hominina .
>
> ex:Hominina a owl:Class ;
>     rdfs:label "Hominina" ;
>     rdfs:comment "A subtribe containing Homo sapiens." ;
>     rdfs:subClassOf ex:Hominidae .
>
> ex:Hominidae a owl:Class ;
>     rdfs:label "Hominidae" ;
>     rdfs:comment "The family containing Hominina." ;
>     rdfs:subClassOf ex:Haplorhini .
>
> ex:Haplorhini a owl:Class ;
>     rdfs:label "Haplorhini" ;
>     rdfs:comment "A taxonomic group containing Hominidae." .
>
> ex:Primates a owl:Class ;
>     rdfs:label "Primates" ;
>     rdfs:comment "An order of mammals containing Hominidae." ;
>     rdfs:subClassOf ex:Mammalia .
>
> ex:Mammalia a owl:Class ;
>     rdfs:label "Mammalia" ;
>     rdfs:comment "Class of mammals." ;
>     rdfs:subClassOf ex:Animal .
>
> ex:Animal a owl:Class ;
>     rdfs:label "Animal" ;
>     rdfs:comment "A living organism capable of sensation and voluntary movement." .
>
> ex:Homo_sapiens a rdfs:subClassOf ex:Animal .
>
> ex:Modern_people a owl:Class ;
>     rdfs:label "Modern people" ;
>     rdfs:comment "Refers to people living today, classified as Homo sapiens." ;
>     rdfs:subClassOf ex:Homo_sapiens .
>
> ex:John a ex:Person .
>
> # End of representation
> ```
> Let me check a few more things:
>
> - The text says "Hominidae contains Hominina", which is represented by `Hominina rdfs:subClassOf Hominidae`.
> - Correct.
> - The text says "Every Hominidae is a Haplorhini", so `Hominidae rdfs:subClassOf Haplorhini`.
> - Correct.
>
> Let me also ensure that the class definitions are consistent and that each comment is correct.
>
> Let me also ensure that I am not minting more than necessary. For example, "Modern people" is a class here, which is appropriate since it appears in the text. I'm not introducing any helper relations or abstractions unless needed.
>
> Let me also ensure that the ontologies are used correctly:
> - `rdfs:subClassOf` is used for taxonomic hierarchies, which is appropriate.
> - All minted terms under `ex:` are documented with labels and comments.
>
> Let me also represent the statement "Nobody argues with the idea that primates are mammals, yet some people take umbrage with the idea that *homo sapiens* is an animal." The controversy is not modeled as a fact, but the text does assert that *homo sapiens* is an animal, so that is included in the RDF.
>
> With this, the representation is complete and faithful to the text.
>
> Let me return the final Turtle as a fenced block. I will not include any explanatory text outside of it, as per the instructions.


### 2.5. Parseability Gate

Before we can talk about overlap, missing facts, or extra facts, we first need a graph.
The next cell extracts Turtle blocks from the final answer and asks RDFLib to parse them.

If parsing fails, that is not just a notebook inconvenience.
For an integrated system, a malformed final serialization is itself a failed output condition.


In [7]:
import mistune


def extract_turtle_code(markdown_text: str) -> list[str]:
    # 1. Initialize Mistune to return an Abstract Syntax Tree (AST)
    # Using 'ast' is much faster than rendering to HTML
    markdown = mistune.create_markdown(renderer="ast")
    tokens = markdown(markdown_text)

    turtle_blocks = []

    # 2. Iterate through tokens to find fenced code blocks
    for token in tokens:
        # Check if the block is a code block and has the 'turtle' info string
        if token["type"] == "block_code":
            if token.get("attrs", {}).get("info", None) in {"text/turtle", "turtle"}:
                turtle_blocks.append(token["raw"].strip())

    return turtle_blocks


def extract_graphs(markdown_text: str) -> list[Graph]:
    return [
        Graph().parse(data=turtle_block, format="turtle")
        for turtle_block in extract_turtle_code(markdown_text)
    ]


def extract_graph_union(markdown_text: str) -> Graph:
    return sum(extract_graphs(markdown_text), Graph())


message = agent_response["messages"][-1]
actual_graph: Graph | None = None
parse_error: Exception | None = None

try:
    actual_graph = extract_graph_union(message.content)
except Exception as e:
    parse_error = e
    print(f"Parseability gate failed: {e}")

Parseability gate failed: at line 2 of <>:
Bad syntax (Prefix "rdfs:" not bound) at ^ in:
"b'<urn:ex:John> a <urn:ex:Person> ;\n    '^b'rdfs:label "John" .\n\n<urn:ex:Homo_sapiens> a owl:Class ;\n   '..."


### 2.6. Conditional Metrics

Semantic metrics are only meaningful after the parseability gate passes.
When we do have a valid graph, we can compare plain triple sets directly because this target graph contains no blank nodes.

That still leaves two interpretation issues:

1. Missing target triples usually indicate a modeling or grounding problem.
2. Extra triples are sometimes harmless documentation and sometimes genuine errors.

The following notes explain how to read those cases when a run produces parseable RDF.

In [8]:
from demo_utils import evaluate_actual_graph

intersection = Graph()
missing = Graph()
extra = Graph()
union = Graph()

if actual_graph is None:
    measures = None
    graphs = None
else:
    measures, graphs = evaluate_actual_graph(actual_graph)
    intersection = graphs["intersection"]
    missing = graphs["missing"]
    extra = graphs["extra"]
    union = graphs["union"]

#### 2.6.1. Intersection with Target

In [9]:
import sys

if measures is None:
    print("Semantic metrics unavailable because the parseability gate failed.")
elif measures["counts"]["intersection"] > 0:
    print(f"{len(intersection)} triples in the intersection")
    print("-" * 80)
    print(intersection.serialize(format="turtle"))
else:
    print(
        "The agent's graph output did NOT intersect the expected graph.",
        file=sys.stderr,
    )

Semantic metrics unavailable because the parseability gate failed.


#### 2.6.2. Interpreting Missing Facts

When a parsed graph is missing target triples, several things may have happened.

- The Research Agent may have minted local terms instead of reusing the target vocabulary.
- It may have chosen a different but still partly intelligible modeling pattern.
- It may simply have omitted content that should have been represented.

These cases are not equivalent, which is why overlap metrics need qualitative interpretation alongside the counts.
They also motivate later middleware that helps the Research Agent inspect or retrieve vocabulary information instead of relying only on baseline model memory.

In [10]:
if measures is None:
    print("No missing-triples report because the parseability gate failed.")
else:
    print(missing.serialize(format="turtle"))

No missing-triples report because the parseability gate failed.


#### 2.6.3. Interpreting Extra Facts

Extra triples are not automatically bad.
A parsed graph may contain documentation triples or other defensible details that are absent from the minimal target graph.

At the same time, extra triples can also reveal genuine semantic mistakes, such as using a familiar RDF term incorrectly or reversing a relation's intended direction.
This is why simple set-based metrics are helpful but not sufficient on their own.

In [11]:
if measures is None:
    print("No extra-triples report because the parseability gate failed.")
else:
    print(extra.serialize(format="turtle"))

No extra-triples report because the parseability gate failed.


#### 2.6.4. What This Means for Evaluation

This baseline is useful, but only with explicit quality gates.

The practical evaluation order is:

1. Check whether the final Turtle block exists and parses.
2. If parsing succeeds, compute overlap metrics.
3. Interpret missing and extra triples manually enough to distinguish harmless variation from semantic failure.

The current checked-in run fails at step 1, which means semantic metrics are unavailable for that run.
That is still an informative result: direct serialization is brittle enough that parseability must be treated as its own baseline outcome.

This also clarifies the comparison with `DatasetMiddleware`.
When middleware moves graph construction into a structured side channel and serializes from middleware-owned state, improvements in syntax robustness and parseability are part of the measured benefit.


In [12]:
if measures is None:
    print("Semantic metrics unavailable because the parseability gate failed.")
else:
    expected_count = measures["counts"]["expected"]
    actual_count = measures["counts"]["extracted"]
    intersection_count = measures["counts"]["intersection"]
    missing_count = measures["counts"]["false_negatives"]
    extra_count = measures["counts"]["false_positives"]
    union_count = measures["counts"]["union_count"]

    precision = measures["metrics"]["precision"]
    recall = measures["metrics"]["recall"]
    f1_score = measures["metrics"]["f1_score"]
    jaccard_index = measures["metrics"]["jaccard_index"]
    triple_edit_distance = measures["metrics"]["triple_edit_distance"]
    normalized_triple_edit_distance = measures["metrics"][
        "normalized_triple_edit_distance"
    ]
    exact_match = measures["exact_match"]

    print(f"Expected triples: {expected_count}")
    print(f"Actual triples: {actual_count}")
    print(f"Intersection triples: {intersection_count}")
    print(f"Missing triples: {missing_count}")
    print(f"Extra triples: {extra_count}")
    print(f"Union triples: {union_count}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"F1 score: {f1_score:.3f}")
    print(f"Jaccard Index: {jaccard_index:.3f}")
    print(f"Triple Edit Distance: {triple_edit_distance}")
    print(f"Normalized Triple Edit Distance: {normalized_triple_edit_distance:.3f}")
    print(f"Exact Match: {exact_match}")

Semantic metrics unavailable because the parseability gate failed.


## 3. Conclusions

This notebook serves a narrow purpose: it shows what a prompt-only RDF extraction baseline looks like before dataset middleware is added.

In the checked-in run, the baseline does **not** clear the parseability gate.
That means we cannot compute semantic overlap metrics for this output, and that limitation is itself part of the result rather than an incidental notebook failure.

The prompt supplements used here are best understood as tool-agnostic guidance extracted from middleware prompts.
They reduce prompt asymmetry, but they do not reproduce the middleware's structured datastore side channel.

That is why improved syntax robustness in the middleware condition counts as a real positive result.
If `DatasetMiddleware` produces parseable RDF more reliably, that belongs in the comparison.

[The next notebook](./demo-dataset-middleware.ipynb) examines that tool-enabled condition directly.